# All-Customer Activity Clustering Alternative

This notebook tests an unsupervised segmentation for every study customer. It uses only available activity features and does not impute missing spending or invent booking recency for customers without bookings.

This is an **alternative to compare**, not an automatic replacement for the RFM analysis. Monetary value and Length remain profiling dimensions with explicit coverage. The candidate labels are arbitrary identifiers, not ranked tiers or traveler personas.

Input: `data/own_processed/user_features_windowed.csv`.

Outputs: `reports/full_cohort_clustering/`. Source data, earlier RFM results and perk assignments are never changed. Repeated runs replace only this notebook's own derived outputs.

Dependencies: pandas and NumPy. No database connection or additional package installation is needed.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

print("STEP 1: Load the study feature table", flush=True)
PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "data" / "own_processed" / "user_features_windowed.csv").is_file()), None)
if PROJECT is None:
    raise FileNotFoundError("Expected data/own_processed/user_features_windowed.csv. No output written.")
SOURCE = PROJECT / "data" / "own_processed" / "user_features_windowed.csv"
OUT = PROJECT / "reports" / "full_cohort_clustering"
d = pd.read_csv(SOURCE, dtype={"user_id": "string"})
assert d.user_id.is_unique and len(d) > 1000
# Available for every customer; no invented booking recency or missing spending.
features = ["session_recency_days", "rfm_frequency_trips"]
values = d[features].to_numpy(float)
assert np.isfinite(values).all() and (values >= 0).all()
Z = np.log1p(values)
mu, sigma = Z.mean(axis=0), Z.std(axis=0)
assert (sigma > 0).all()
X = (Z - mu) / sigma
def fit_once(x,k,rng,max_iter=300):
 n=len(x); centers=[x[rng.integers(n)].copy()]
 best=np.sum((x-centers[0])**2,axis=1)
 for j in range(1,k):
  idx=rng.choice(n,p=best/best.sum()) if best.sum()>0 else rng.integers(n)
  centers.append(x[idx].copy());best=np.minimum(best,np.sum((x-centers[-1])**2,axis=1))
 centers=np.array(centers);previous=None
 for it in range(max_iter):
  dist=np.sum((x[:,None,:]-centers[None,:,:])**2,axis=2);lab=dist.argmin(axis=1)
  if previous is not None and np.array_equal(lab,previous):break
  previous=lab.copy()
  for j in range(k):
   if np.any(lab==j):centers[j]=x[lab==j].mean(axis=0)
   else: centers[j]=x[np.argmax(dist.min(axis=1))]
 inertia=float(np.sum((x-centers[lab])**2));return lab,centers,inertia
def fit(x,k,seed):
 rng=np.random.default_rng(seed);best=None
 for _ in range(10):
  result=fit_once(x,k,rng)
  if best is None or result[2]<best[2]:best=result
 return best
# Agreement invariant to arbitrary label numbering.
def ari(a,b):
 _,ai=np.unique(a,return_inverse=True);_,bi=np.unique(b,return_inverse=True)
 mat=np.zeros((ai.max()+1,bi.max()+1),dtype=np.int64);np.add.at(mat,(ai,bi),1)
 comb=lambda arr:float(np.sum(arr*(arr-1)/2))
 observed=comb(mat);ar=comb(mat.sum(axis=1));br=comb(mat.sum(axis=0));pairs=len(a)*(len(a)-1)/2
 expected=ar*br/pairs;den=(ar+br)/2-expected
 return (observed-expected)/den if den else 1.0
sample=np.random.default_rng(2026).choice(len(d),1000,replace=False)
def sil(x,labels):
 xx=x[sample];ll=labels[sample];dist=np.sqrt(np.sum((xx[:,None,:]-xx[None,:,:])**2,axis=2))
 aa=np.zeros(len(xx));bb=np.full(len(xx),np.inf);single=np.zeros(len(xx),bool)
 for c in np.unique(ll):
  ix=np.where(ll==c)[0];outside=np.where(ll!=c)[0]
  single[ix]=len(ix)==1
  if len(ix)>1:aa[ix]=dist[np.ix_(ix,ix)].sum(axis=1)/(len(ix)-1)
  bb[outside]=np.minimum(bb[outside],dist[np.ix_(outside,ix)].mean(axis=1))
 vals=(bb-aa)/np.maximum(aa,bb);vals[single]=0
 return float(np.nanmean(vals))

print("STEP 2: Test 2–6 all-customer K-means groups", flush=True)
metrics = []
labels_by_k = {}
for k in range(2, 7):
    print(f"  Fitting {k} groups", flush=True)
    labels, centers, inertia = fit(X, k, 42)
    labels_by_k[k] = labels
    counts = np.bincount(labels, minlength=k)
    agreement = [ari(labels, fit(X,k,seed)[0]) for seed in [43,44]]
    check = sum(float(((X[labels==j] - X[labels==j].mean(axis=0))**2).sum()) for j in range(k))
    assert abs(check - inertia) < 1e-5
    metrics.append({"groups": k, "sample_silhouette": sil(X,labels),
                    "minimum_seed_ARI": min(agreement), "smallest_group": int(counts.min()),
                    "largest_group": int(counts.max()), "inertia": inertia})
metric_table = pd.DataFrame(metrics)

print("STEP 3: Profile all candidates without defining personas", flush=True)
profile_features = features + ["average_session_seconds", "num_recent_sessions", "average_page_clicks", "rfm_monetary_estimate",
    "rfm_recency_days", "lrfm_length_days", "recent_potential_exposure_days",
    "average_checked_bags", "average_known_hotel_nights", "flight_discounted_booking_rate",
    "hotel_discounted_booking_rate", "cancellation_rate", "net_booking_conversion"]
profile_rows, status_rows = [], []
assignments = d[["user_id", "baseline_status", "booking_status"]].copy()
for k, labels in labels_by_k.items():
    assignments[f"activity_k{k}"] = labels + 1
    for group in range(k):
        sub = d.loc[labels == group]
        for col in profile_features:
            observed = pd.to_numeric(sub[col], errors="coerce").dropna()
            profile_rows.append({"groups": k, "cluster": group+1, "feature": col,
                "customers": len(sub), "customers_with_value": len(observed),
                "coverage_share": len(observed)/len(sub), "median": observed.median(), "mean": observed.mean()})
        for status,count in sub.booking_status.value_counts().items():
            status_rows.append({"groups":k,"cluster":group+1,"booking_status":status,
                               "customers":int(count),"share_of_cluster":int(count)/len(sub)})
assert assignments.filter(like="activity_k").notna().all().all()

print("STEP 4: Save the comparison and limitations", flush=True)
OUT.mkdir(parents=True, exist_ok=True)
metric_table.to_csv(OUT / "full_cohort_cluster_metrics.csv", index=False)
pd.DataFrame(profile_rows).to_csv(OUT / "full_cohort_cluster_profiles.csv", index=False)
pd.DataFrame(status_rows).to_csv(OUT / "full_cohort_booking_status_profiles.csv", index=False)
assignments.to_csv(OUT / "full_cohort_candidate_assignments.csv", index=False)
pd.DataFrame({"feature":features,"log1p_mean":mu,"log1p_population_std":sigma}).to_csv(OUT / "full_cohort_scaling_parameters.csv",index=False)
best = int(metric_table.loc[metric_table.sample_silhouette.idxmax(), "groups"])
text = ["# Full-Cohort Activity Clustering: Candidate Review", "",
        f"Source: {SOURCE.name}. All {len(d):,} study customers are included in every candidate solution.",
        "", "## Purpose", "Test a clustering-only alternative that can include customers without bookings or complete spending. This does not replace or finalize the existing RFM model automatically.",
        "", "## Model definition", "Inputs: engagement recency (days since last session), noncanceled trip frequency in January 5–July 28, 2023. Session duration is retained for profiling rather than used to drive clusters. Each input uses log1p followed by population-standard-deviation scaling. K-means uses K-means++ initialization, 10 starts and up to 300 iterations. Main seed 42; sensitivity seeds 43 and 44. Silhouette is computed on the same fixed 1,000-customer sample for every candidate (seed 2026).",
        "", "This is activity/engagement clustering, NOT classic RFM or LRFM. Booking spending, booking recency and Length are retained for profiling with explicit coverage; undefined or unknown values are not invented. Actual zero recent bookings is retained as zero.",
        "", "## Validation results", "```", metric_table.round(4).to_string(index=False), "```",
        f"The strongest sample silhouette in this model family occurs at {best} groups. Business usefulness and stable interpretation still require reviewing profiles.",
        "", "## Four-group candidate profile", "Four groups are shown for comparison with the earlier RFM candidate, not imposed as the final answer.", "```"]
view = pd.DataFrame(profile_rows)
view = view.loc[view.groups.eq(4) & view.feature.isin(features + ["rfm_monetary_estimate", "recent_potential_exposure_days"])]
text += [view.pivot(index="cluster",columns="feature",values="median").round(2).to_string(), "```",
         "", "## Feature-selection note", "An exploratory version that also included mean session duration isolated a long-session group. Duration is omitted from the main candidate because its traveler-need interpretation is unsupported; it remains a profiling variable. This feature choice is an analyst judgment, not a validated optimal specification.", "", "## Limits and model-selection safeguards", "Engagement recency is not booking recency. Average session duration may reflect generated-data artifacts and cannot establish decisiveness, uncertainty or business travel. Feature windows differ and are explicitly labeled. Customer signup exposure remains unequal. Silhouette values from this all-customer feature space are not a valid direct ranking against the earlier RFM complete-booker model. Seed agreement is not bootstrap or future-period stability. No actual future outcomes or perk responses are available. Models are compared by coverage, internal validation, original-unit profiles and business usefulness, not by declaring any framework universally superior.",
         "", "## Next decision", "Choose between retaining RFM as the primary booker analysis with separately described no-booking handling, or using this full-cohort activity clustering as the main segmentation and RFM/LRFM as profile dimensions. Preserve all candidate results. Continue to treat perk assignments as a separate pilot policy, not empirically identified favorite perks. Mention all final groups and highlight approximately three."]
(OUT / "full_cohort_clustering_summary.md").write_text("\n".join(text),encoding="utf-8")
print("COMPLETE:", OUT, flush=True)
print("Earlier clustering and perk outputs were not changed.", flush=True)
